# PRISM — Inverse: MAP and MAP + Laplace

Needs the PRISM and CNN checkpoints: on Kaggle attach the outputs of notebooks 02 and 04. `target_n=None` inverts all held-out test stars. Always runs in full fp32.

Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**. Inputs: the `aspcapstar-dr17-150kstars` and `element-masks` datasets.

In [ ]:
# Install PRISM without touching Kaggle's preinstalled JAX / Flax / Orbax versions.
# PRISM_REF: the git branch / tag / commit to install (the branch holding the prism/ package).
# Running locally? Skip this cell and use the repo's .venv as the kernel.
PRISM_REF = "main"
!pip freeze | grep -E '^(jax|jaxlib|jax-cuda|flax|orbax|optax|numpy|scipy)' > /tmp/kaggle-pins.txt
!pip install -q -c /tmp/kaggle-pins.txt "git+https://github.com/Aneeshshastri/PRISM@{PRISM_REF}" blackjax

In [ ]:
from prism.env import check_env
check_env()  # Kaggle: must report 2 GPUs (cc 7.5). Local single GPU: set PRISM_EXPECTED_GPUS=1

In [ ]:
from prism import stages
from prism.config import Config

# Finds the APOGEE HDF5 and element-masks datasets under /kaggle/input (any mount layout).
# Locally, set PRISM_INPUT_ROOT / PRISM_CACHE_DIR / PRISM_ARTIFACT_DIR (see README).
cfg = Config.for_kaggle(checkpoint_inputs="/kaggle/input/**/prism_artifacts/checkpoints")
# Override anything, e.g. Config.for_kaggle(batch_size=512, train_precision='bfloat16')

In [ ]:
stages.run_inverse(cfg, 'map', target_n=None)
stages.run_inverse(cfg, 'laplace', target_n=None)

In [ ]:
!find {cfg.artifact_dir} -maxdepth 3 | head -60